# Load Libraries

In [59]:
import os
import re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
import loguru as logger

from settings.settings_analyze_efizz import Settings_ae
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.rayleigh.load_rayleigh import collect_all_rayleigh_paths, load_all_rayleigh_data
from behave_analysis.utils.rayleigh.manipulate_rayleigh_df import extract_compartment_values, extract_firing_rates
from behave_analysis.utils.creating_directories import make_directory
from settings.settings_analyze_efizz import Settings_ae as Settings
from behave_analysis.analyze.TunED.model import TunEdModel


# Hard code data to load

In [13]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_shelt_4mar
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

In [14]:
# JAL6
experiments_objects = [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

# Regex to clean up .arrow file names

In [15]:
def regex(angle):
    "Remove unwanted characters from angle file string"
    pattern = r'^(.*?)(?=_Rayleigh)'
    match = re.search(pattern, angle)
    assert match, f"Could not find match for {angle}"
    return match.group(0)

# init parmas

In [16]:
# Init Params
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
total_cells = 0
total_sessions = 0
rayleigh_threshold = 0.15
fr_threshold = 5 # Hz
dir = make_directory(r"Z:\Jasmine_Laurence\rayleigh_analysis")

# Loop over sessions and count which angle / condition has the highest rayleigh vector by compartment

In [6]:
nested_across_session_ray_data = defaultdict(lambda: defaultdict(int)) # all_compartment_ray_data[session][cell] = {max_shelter_rayleigh_name, max_threat_rayleigh_name, max_shelter_rayleigh, max_threat_rayleigh}
for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    total_sessions += 1
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    total_cells += nCells
    
    for cell in range(nCells):
        rayleigh_shelter = 0
        rayleigh_threat = 0
        for ci, condition in enumerate(condition_data.keys()):
            for angle in condition_data[condition].keys():
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh")

                # Replace the largest rayleigh value if it is larger than the current one AND larger than the threshold and save condition | angle combo for that cell
                # Threat compartment
                if np.logical_and(output[cell][0] > rayleigh_shelter, output[cell][0] > rayleigh_threshold): # first element is shelter only
                    rayleigh_shelter = output[cell][0]
                    max_shelter_angle_str = regex(angle) + " | " + condition
                    #max_shelter_angle_str = angle
                    #cell_id = condition_data[condition][angle][cell]["clusterID"][0] # Get the cell ID

                # Shelter compartment
                if np.logical_and(output[cell][1] > rayleigh_threat, output[cell][1] > rayleigh_threshold): # second element is threat only
                    rayleigh_threat = output[cell][1]
                    max_threat_angle_str = regex(angle) + " | " + condition
                    
        
        # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
        try:
            nested_across_session_ray_data[i][cell] = {"max_shelter_rayleigh_name": max_shelter_angle_str, 
                                    "max_threat_rayleigh_name": max_threat_angle_str,
                                    "max_shelter_rayleigh": rayleigh_shelter,
                                    "max_threat_rayleigh": rayleigh_threat}
        except:
            print(f"Cell {cell} in session {i} did not meet threshold")

# Across compartment and all conditions

In [33]:
all_compartment_ray_data = defaultdict(lambda: defaultdict(int)) # nested_across_session_ray_data[session][cell] = {max_shelter_rayleigh_name, max_threat_rayleigh_name, max_shelter_rayleigh, max_threat_rayleigh}
#test = nest_dic()
for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    total_sessions += 1
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    total_cells += nCells

    for cell in range(nCells):
        rayleigh = 0
        for ci, condition in enumerate(condition_data.keys()):
            for angle in condition_data[condition].keys():
                output = condition_data[condition][angle]["arena_rayleigh"]

                if np.logical_and(output[cell] > rayleigh, output[cell] > rayleigh_threshold): # second element is threat only
                    rayleigh = output[cell]
                    max_angle_str = regex(angle) + " | " + condition
                    
        # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
        try:
            all_compartment_ray_data[i][cell] = {"max_rayleigh_name": max_angle_str}
                            
        except:
            print(f"Cell {cell} in session {i} did not meet threshold")

Cell 0 in session 0 did not meet threshold


# Data per session and cell

In [24]:
remove = ['h_bar_centre_a | barrier_post_flip', 
          'h_bar_centre_a | shelter_only', 
          'h_bar_centre_a | barrier_pre_flip',
          'hdir | barrier_post_flip',
          'hdir | shelter_only',
          'hdir | barrier_pre_flip',]

remove = ['h_bar_centre_a | barrier_post_flip', 
          'h_bar_centre_a | shelter_only', 
          'h_bar_centre_a | barrier_pre_flip',]

# Mouse session map

In [19]:
# Mice groups based on session names
mice_groups = {
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept', 'JAL005_8thSept', 'JAL005_21stSept'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}


session_names = ["JAL6_flip7_1apr", "JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept", "JAL005_8thSept", "JAL005_21stSept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

# Plot fraction of cells across sessions and average mice for just threat zone

In [52]:
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import numpy as np

# Initialize a set to collect all unique 'max_threat_rayleigh_name' across all sessions
all_unique_names = set()

# First pass to collect all unique names
for session in nested_across_session_ray_data:
    session_data = nested_across_session_ray_data[session]
    for cell in session_data:
        rayleigh_name = session_data[cell]['max_threat_rayleigh_name']
        if rayleigh_name not in remove:
            all_unique_names.add(rayleigh_name)

# Convert to sorted list for consistent ordering
sorted_names = sorted(all_unique_names)

# Initialize a dictionary to store all session fractions for each mouse
mice_session_fractions = defaultdict(list)

# Collect fractions for each session
for session, session_name in zip(nested_across_session_ray_data, session_names):
    session_data = nested_across_session_ray_data[session]
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
        
    if not mouse:
        raise ValueError("No mouse found for session: " + session_name)  # Raise an error if no mouse is found
    
    # Total number of cells in the session
    total_cells = len(session_data)

    # Collect all 'max_threat_rayleigh_name' for the session, excluding 'remove'
    threat_rayleigh_names = [
        session_data[cell]['max_threat_rayleigh_name'] for cell in session_data 
        if session_data[cell]['max_threat_rayleigh_name'] not in remove
    ]
    
    # Count occurrences of each 'max_threat_rayleigh_name'
    name_counts = Counter(threat_rayleigh_names)
    
    # Calculate the fraction of cells for each name in sorted_names
    session_fractions = [name_counts.get(name, 0) / total_cells for name in sorted_names]
    
    # Store fractions for the current mouse
    mice_session_fractions[mouse].append(session_fractions)

# Calculate the overall mean for each 'max_threat_rayleigh_name' across all mice
mean_fractions_all_mice = np.mean(
    [np.mean(mice_session_fractions[mouse], axis=0) for mouse in mice_session_fractions], 
    axis=0
)

# Sort the names by their mean fraction in descending order
sorted_indices = np.argsort(-mean_fractions_all_mice)
sorted_names = [sorted_names[i] for i in sorted_indices]

# Update the plot with the new sorted order
plt.figure(figsize=(12, 8))

# Plot each individual session in grey
for session in nested_across_session_ray_data:
    session_data = nested_across_session_ray_data[session]  
    total_cells = len(session_data)

    # Collect all 'max_threat_rayleigh_name' for the session, excluding 'remove'
    threat_rayleigh_names = [
        session_data[cell]['max_threat_rayleigh_name'] for cell in session_data 
        if session_data[cell]['max_threat_rayleigh_name'] not in remove
    ]
    
    # Count occurrences of each 'max_threat_rayleigh_name'
    name_counts = Counter(threat_rayleigh_names)
    
    # Calculate the fraction of cells for each name in sorted_names (now sorted by mean)
    session_fractions = [name_counts.get(name, 0) / total_cells for name in sorted_names]
    
    # Plot the individual session in grey
    plt.plot(session_fractions, sorted_names, color='grey', alpha=0.5, linewidth=1)

# Calculate and plot the average line for each mouse in distinct colors
colors = plt.cm.get_cmap('tab10', len(mice_session_fractions))  # Use a distinct color for each mouse
for i, (mouse, session_fractions_list) in enumerate(mice_session_fractions.items()):
    # Calculate the mean across sessions for this mouse
    mean_fractions = np.mean(session_fractions_list, axis=0)
    
    # Sort the mean fractions in the same order as sorted_names
    sorted_mean_fractions = [mean_fractions[idx] for idx in sorted_indices]
    
    # Plot the average line for the mouse
    plt.plot(sorted_mean_fractions, sorted_names, marker='o', linestyle='-', color=colors(i), linewidth=2.5, label=f'{mouse} Average')

# Add labels and title
plt.xlabel('Fraction of Cells', fontsize = 14)
plt.ylabel('Max rayleigh vector combination', fontsize=16)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# Increase space between labels and show the plot
plt.tight_layout()
plt.show()


# Now across arena

In [45]:
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import numpy as np

# Initialize a set to collect all unique 'max_threat_rayleigh_name' across all sessions
all_unique_names = set()

# First pass to collect all unique names
for session in all_compartment_ray_data:
    session_data = all_compartment_ray_data[session]
    for cell in session_data:
        rayleigh_name = session_data[cell]['max_rayleigh_name']
        if rayleigh_name not in remove:
            all_unique_names.add(rayleigh_name)

# Convert to sorted list for consistent ordering
sorted_names = sorted(all_unique_names)

# Initialize a dictionary to store all session fractions for each mouse
mice_session_fractions = defaultdict(list)

# Collect fractions for each session
for session, session_name in zip(all_compartment_ray_data, session_names):
    session_data = all_compartment_ray_data[session]
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
        
    if not mouse:
        raise ValueError("No mouse found for session: " + session_name)  # Raise an error if no mouse is found
    
    # Total number of cells in the session
    total_cells = len(session_data)

    # Collect all 'max_threat_rayleigh_name' for the session, excluding 'remove'
    threat_rayleigh_names = [
        session_data[cell]['max_rayleigh_name'] for cell in session_data 
       if session_data[cell]['max_rayleigh_name'] not in remove
    ]
    
    # Count occurrences of each 'max_threat_rayleigh_name'
    name_counts = Counter(threat_rayleigh_names)
    
    # Calculate the fraction of cells for each name in sorted_names
    session_fractions = [name_counts.get(name, 0) / total_cells for name in sorted_names]
    
    # Store fractions for the current mouse
    mice_session_fractions[mouse].append(session_fractions)

# Calculate the overall mean for each 'max_threat_rayleigh_name' across all mice
mean_fractions_all_mice = np.mean(
    [np.mean(mice_session_fractions[mouse], axis=0) for mouse in mice_session_fractions], 
    axis=0
)

# Sort the names by their mean fraction in descending order
sorted_indices = np.argsort(-mean_fractions_all_mice)
sorted_names = [sorted_names[i] for i in sorted_indices]

# Update the plot with the new sorted order
plt.figure(figsize=(12, 8))

# Plot each individual session in grey
for session, session_name in zip(all_compartment_ray_data, session_names):
    session_data = all_compartment_ray_data[session]  
    total_cells = len(session_data)

    # Collect all 'max_threat_rayleigh_name' for the session, excluding 'remove'
    threat_rayleigh_names = [
        session_data[cell]['max_rayleigh_name'] for cell in session_data 
       if session_data[cell]['max_rayleigh_name'] not in remove
    ]
    
    # Count occurrences of each 'max_threat_rayleigh_name'
    name_counts = Counter(threat_rayleigh_names)
    
    # Calculate the fraction of cells for each name in sorted_names (now sorted by mean)
    session_fractions = [name_counts.get(name, 0) / total_cells for name in sorted_names]
        
    # Plot the individual session in grey
    plt.plot(session_fractions, sorted_names, color='grey', alpha=0.5, linewidth=1)

# Calculate and plot the average line for each mouse in distinct colors
colors = plt.cm.get_cmap('tab10', len(mice_session_fractions))  # Use a distinct color for each mouse
for i, (mouse, session_fractions_list) in enumerate(mice_session_fractions.items()):
    # Calculate the mean across sessions for this mouse
    mean_fractions = np.mean(session_fractions_list, axis=0)
    
    # Sort the mean fractions in the same order as sorted_names
    sorted_mean_fractions = [mean_fractions[idx] for idx in sorted_indices]
    
    # Plot the average line for the mouse
    plt.plot(sorted_mean_fractions, sorted_names, marker='o', linestyle='-', color=colors(i), linewidth=2.5, label=f'{mouse} Average')

# Add labels and title
plt.xlabel('Fraction of Cells', fontsize = 14)
plt.ylabel('Max rayleigh vector combination', fontsize=16)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# Increase space between labels and show the plot
plt.tight_layout()
plt.show()

# Same logic for each condition

In [52]:
def nest_dic():
    """A function to create arbitrarily nested dictionaries"""
    return defaultdict(nest_dic)

In [115]:
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

In [229]:
# TODO firing rate threshold not implemented

dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}

cell_count = 0
# For each experiment object 
for i, session in enumerate(experiments_objects):
    
    # load the session
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # Count the number of sessions and cells
    total_sessions += 1
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0
            for angle in angle_keys:
                
                #Just the threat zone
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for each compartment
                if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                    rayleigh = output[cell][1]
                    max_angle_str = regex(angle)                

                # # Whole arena rayleigh
                # output = condition_data[condition][angle]["arena_rayleigh"] # Whole arena rayleigh
                # if np.logical_and(output[cell] > rayleigh, output[cell] > rayleigh_threshold):
                #     rayleigh = output[cell]
                #     max_angle_str = regex(angle)
                    
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                dict[i][cell][condition] = max_angle_str
                                
            except:
                print(f"Cell {cell} in session {i} did not meet threshold")

In [225]:
cell_count

4858

# Across session and mice counts

In [230]:
# Counts across sessions
shelter = []
barrier_pre_flip = []
barrier_post_flip = []
for session in dict:
    for cell in dict[session]:
        x = dict[session][cell]["shelter_only"]
        shelter.append(x)
        y = dict[session][cell]["barrier_pre_flip"]
        barrier_pre_flip.append(y)
        z = dict[session][cell]["barrier_post_flip"]
        barrier_post_flip.append(z)
shelter_counts = Counter(shelter)
barrier_pre_flip_counts = Counter(barrier_pre_flip)
barrier_post_flip_counts = Counter(barrier_post_flip)
print("Across session counts")
print(shelter_counts)

Across session counts
Counter({'h_postflipbar_a': 1353, 'h_preflipbar_a': 1321, 'hdir': 1209, 'hsa': 975})


In [237]:
# Counts within sessions
within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

for session in dict:
    for cell in dict[session]:
        x = dict[session][cell]["shelter_only"]
        y = dict[session][cell]["barrier_pre_flip"]
        z = dict[session][cell]["barrier_post_flip"]
        within_session_counts_shelter[session][x] += 1
        within_session_counts_bar_pre_flip[session][y] += 1
        within_session_counts_bar_post_flip[session][z] += 1



defaultdict(int,
            {'hsa': 47,
             'hdir': 75,
             'h_postflipbar_a': 84,
             'h_preflipbar_a': 65})

# Plot a bar plot

In [258]:
xcoords = [0, 1, 3, 4]
plt.bar(xcoords, 
        [barrier_pre_flip_counts['h_preflipbar_a'] / cell_count, 
         barrier_pre_flip_counts['h_postflipbar_a'] / cell_count, 
         barrier_post_flip_counts['h_preflipbar_a'] / cell_count, 
         barrier_post_flip_counts['h_postflipbar_a'] / cell_count],
        color= 'darkorchid',
        alpha = 0.75
        )
plt.xticks(xcoords, 
           ['Open Edge | Pre flip condition', 
            'Closed Edge | Pre flip condition', 
            'Closed Edge | Post flip condition', 
            'Open Edge | Post flip condition'],
           rotation=10)
plt.ylabel('Fraction of cells', fontsize=16)

# Counts within sessions
within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

for session in dict:
    for cell in dict[session]:
        x = dict[session][cell]["shelter_only"]
        y = dict[session][cell]["barrier_pre_flip"]
        z = dict[session][cell]["barrier_post_flip"]
        within_session_counts_shelter[session][x] += 1
        within_session_counts_bar_pre_flip[session][y] += 1
        within_session_counts_bar_post_flip[session][z] += 1
            
    # Scatter plot points
    y_scatter_values = [
        within_session_counts_bar_pre_flip[session]['h_preflipbar_a'] / sum(within_session_counts_bar_pre_flip[session].values()),
        within_session_counts_bar_pre_flip[session]['h_postflipbar_a'] / sum(within_session_counts_bar_pre_flip[session].values()),
        within_session_counts_bar_post_flip[session]['h_preflipbar_a'] / sum(within_session_counts_bar_post_flip[session].values()),
        within_session_counts_bar_post_flip[session]['h_postflipbar_a'] / sum(within_session_counts_bar_post_flip[session].values())
    ]
    
    plt.scatter(xcoords, y_scatter_values, color='grey', s=100, alpha=0.5)
    
    # Draw lines between scatter points for each session
    plt.plot(xcoords[:2], y_scatter_values[:2], color='grey', alpha=0.5)  # Connect points at 0 and 1
    plt.plot(xcoords[2:], y_scatter_values[2:], color='grey', alpha=0.5)  # Connect points at 3 and 4

array = np.array(average_counts)
mean = np.mean(array, axis=0)
print(mean)
                        

        

[0.28972756 0.24496431 0.23476012 0.2876685 ]


# Make line plots

In [62]:
# Initialize a set to collect all unique 'max_rayleigh_name' across all sessions
all_unique_names = set()

# First pass to collect all unique names
for session in dict:
    session_data = dict[session]
    for cell in session_data:
        for condition in session_data[cell]:
            rayleigh_name = session_data[cell][condition]['max_rayleigh_name']
            if rayleigh_name not in remove:
                all_unique_names.add(rayleigh_name)

# Convert to sorted list for consistent ordering
sorted_names = sorted(all_unique_names)

# Initialize a dictionary to store all session fractions for each mouse, split by condition
mice_session_fractions = defaultdict(lambda: defaultdict(list))

# Collect fractions for each session, split by condition
for session, session_name in zip(dict, session_names):
    session_data = dict[session]
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
        
    if not mouse:
        raise ValueError("No mouse found for session: " + session_name)  # Raise an error if no mouse is found
    
    # Total number of cells in the session
    total_cells = len(session_data)

    # Iterate over conditions
    for condition in conditions:
        # Collect all 'max_rayleigh_name' for the session under the current condition, excluding 'remove'
        threat_rayleigh_names = [
            session_data[cell][condition]['max_rayleigh_name'] for cell in session_data 
            if session_data[cell][condition]['max_rayleigh_name'] not in remove
        ]
        
        # Count occurrences of each 'max_rayleigh_name'
        name_counts = Counter(threat_rayleigh_names)
        
        # Calculate the fraction of cells for each name in sorted_names
        session_fractions = [name_counts.get(name, 0) / total_cells for name in sorted_names]
        
        # Store fractions for the current mouse and condition
        mice_session_fractions[mouse][condition].append(session_fractions)

# Calculate the overall mean for each 'max_rayleigh_name' across all mice, for each condition
mean_fractions_all_mice = {
    condition: np.mean(
        [np.mean(mice_session_fractions[mouse][condition], axis=0) for mouse in mice_session_fractions], 
        axis=0
    )
    for condition in conditions
}

# Sort the names by their mean fraction in descending order for each condition
sorted_indices_by_condition = {
    condition: np.argsort(-mean_fractions_all_mice[condition])
    for condition in conditions
}

# Update the plot with the new sorted order
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 18), sharex=True)

# Plot each condition in a separate subplot
colors = plt.cm.get_cmap('tab10', len(mice_session_fractions))  # Use a distinct color for each mouse

for ci, condition in enumerate(conditions):
    ax = axes[ci]
    ax.set_title(f'Condition: {condition}', fontsize=16)
    
    # Get sorted indices and names for the current condition
    sorted_indices = sorted_indices_by_condition[condition]
    sorted_names_condition = [sorted_names[i] for i in sorted_indices]
    
    # Plot each individual session in grey
    for session, session_name in zip(dict, session_names):
        session_data = dict[session]  
        total_cells = len(session_data)

        # Collect all 'max_rayleigh_name' for the session under the current condition, excluding 'remove'
        threat_rayleigh_names = [
            session_data[cell][condition]['max_rayleigh_name'] for cell in session_data 
            if session_data[cell][condition]['max_rayleigh_name'] not in remove
        ]
        
        # Count occurrences of each 'max_rayleigh_name'
        name_counts = Counter(threat_rayleigh_names)
        
        # Calculate the fraction of cells for each name in sorted_names_condition (now sorted by mean)
        session_fractions = [name_counts.get(name, 0) / total_cells for name in sorted_names_condition]
            
        # Plot the individual session in grey
        ax.plot(session_fractions, sorted_names_condition, color='grey', alpha=0.5, linewidth=1)

    # Calculate and plot the average line for each mouse in distinct colors
    for i, (mouse, session_fractions_list) in enumerate(mice_session_fractions.items()):
        # Calculate the mean across sessions for this mouse and condition
        mean_fractions = np.mean(session_fractions_list[condition], axis=0)
        
        # Sort the mean fractions in the same order as sorted_names_condition
        sorted_mean_fractions = [mean_fractions[idx] for idx in sorted_indices]
        
        # Plot the average line for the mouse
        ax.plot(sorted_mean_fractions, sorted_names_condition, marker='o', linestyle='-', color=colors(i), linewidth=2.5, label=f'{mouse} Average')

    # Add labels
    ax.set_ylabel('Max rayleigh vector combination', fontsize=14)
    ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# Add common x label
plt.xlabel('Fraction of Cells', fontsize=14)
plt.tight_layout()
plt.show()

# Make box plots across sessions for the rayleigh value

In [259]:
big_list_shelter = []
big_list_threat = []
session_ids = []
for i, session in enumerate(nested_across_session_ray_data.keys()):
    for cell in nested_across_session_ray_data[i].keys():
        big_list_shelter.append(nested_across_session_ray_data[i][cell]["max_shelter_rayleigh_name"])
        big_list_threat.append(nested_across_session_ray_data[i][cell]["max_threat_rayleigh_name"])
    session_ids.append(i * np.ones(len(nested_across_session_ray_data[i].keys())))

# Counter object creates a dictionary where the keys are the unqiue elements and the values are the counts
shelter_counter = Counter(big_list_shelter)
threat_counter = Counter(big_list_threat)


df = pd.DataFrame(big_list_threat, columns=['condition']) # each row is a condition | angle combo
df['session'] = np.concatenate(session_ids)
df['count'] = 1
group_df = df.groupby(['condition', 'session']).count().reset_index()
mean_counts = group_df.groupby('condition')['count'].mean().sort_values().index
plt.figure(figsize=(10, 6))
sns.boxplot(data=group_df, x='count', y='condition', orient='h', order=mean_counts, flierprops={"marker": "x"}, boxprops={"facecolor": (.4, .6, .8, .5)})
plt.ylabel('Condition | Angle combo')
plt.xlabel('Cell count')
plt.title(f'Total Cells: {total_cells} \n Total Sessions: {total_sessions} \n Compartment: Threat Zone \n Distribution of max rayleigh values across sessions')
plt.show()
#assert group_df['count'].sum() == total_cells, f"Total cells {total_cells} does not match sum of counts {group_df['count'].sum()}"

df = pd.DataFrame(big_list_shelter, columns=['condition']) # each row is a condition | angle combo
df['session'] = np.concatenate(session_ids)
df['count'] = 1
group_df = df.groupby(['condition', 'session']).count().reset_index()
mean_counts = group_df.groupby('condition')['count'].mean().sort_values().index
plt.figure(figsize=(10, 6))
sns.boxplot(data=group_df, x='count', y='condition', orient='h', order=mean_counts, flierprops={"marker": "x"}, boxprops={"facecolor": (.4, .6, .8, .5)})
plt.ylabel('Condition | Angle combo')
plt.xlabel('Cell count')
plt.title(f'Total Cells: {total_cells} \n Total Sessions: {total_sessions} \n Compartment: Shelter Zone \n Distribution of max rayleigh values across sessions')
plt.show()
##assert group_df['count'].sum() == total_cells, f"Total cells {total_cells} does not match sum of counts {group_df['count'].sum()}"



NameError: name 'nested_across_session_ray_data' is not defined